## Background
This Jupiter Notebook aims to set the baseline for manipulation classification task. \
At first data is preprocessed, including lowercasing, lemmatization, removal of emojis, punctuation, and URLs. Then texts are vectorized using two transformer models, i.e., mBERT and XLM-RoBERTa. Then 4 models will be tested: Logistic Regression, Random Forest, XGBoost, and LightGBM. For the last two hyperparameter finetuning will be also applied.

## Imports

In [4]:
import os
import re
import warnings
import emoji
import numpy as np
import pandas as pd
import seaborn as sns
from collections import Counter
import spacy
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.multioutput import MultiOutputClassifier
from transformers import RobertaTokenizerFast, AutoModel, BertTokenizer
import torch
import xgboost as xgb
import lightgbm as lgb

import matplotlib.pyplot as plt

In [5]:
warnings.filterwarnings("ignore")

## Constants

In [ ]:
TRAIN_PATH = "../data/"
TRAIN_NAME = "train.parquet"

In [7]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")

Using device: cuda:0


## Read data

In [8]:
df = pd.read_parquet(os.path.join(TRAIN_PATH, TRAIN_NAME))
df.head()

,id,content,lang,manipulative,techniques,trigger_words
0,0bb0c7fa-101b-4583-a5f9-9d503339141c,Новий огляд мапи DeepState від російського вій...,uk,True,"[euphoria, loaded_language]","[[27, 63], [65, 88], [90, 183], [186, 308]]"
1,7159f802-6f99-4e9d-97bd-6f565a4a0fae,Недавно 95 квартал жёстко поглумился над русск...,ru,True,"[loaded_language, cherry_picking]","[[0, 40], [123, 137], [180, 251], [253, 274]]"
2,e6a427f1-211f-405f-bd8b-70798458d656,🤩\nТим часом йде евакуація Бєлгородського авто...,uk,True,"[loaded_language, euphoria]","[[55, 100]]"
3,1647a352-4cd3-40f6-bfa1-d87d42e34eea,В Україні найближчим часом мають намір посилит...,uk,False,None,None
4,9c01de00-841f-4b50-9407-104e9ffb03bf,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...",ru,True,[loaded_language],"[[114, 144]]"


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3822 entries, 0 to 3821
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             3822 non-null   object
 1   content        3822 non-null   object
 2   lang           3822 non-null   object
 3   manipulative   3822 non-null   bool  
 4   techniques     2589 non-null   object
 5   trigger_words  2589 non-null   object
dtypes: bool(1), object(5)
memory usage: 153.2+ KB


In [10]:
# If there are no manipulations in the text, we will set techniques and trigger_words to empty lists
df['techniques'] = df['techniques'].apply(lambda x: [] if x is None else x)
df['trigger_words'] = df['trigger_words'].apply(lambda x: [] if x is None else x)

In [11]:
# Number of occurrences of each technique
technique_counts = Counter([tech for sublist in df['techniques'] for tech in sublist])
technique_counts

Counter({'euphoria': 462,
         'loaded_language': 1973,
         'cherry_picking': 512,
         'glittering_generalities': 483,
         'cliche': 463,
         'appeal_to_fear': 300,
         'bandwagon': 157,
         'fud': 385,
         'whataboutism': 158,
         'straw_man': 138})

## Data preprocessing

### One-hot encoding of manipulation techniques

In [12]:
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df['techniques'])
Y

array([[0, 0, 0, ..., 1, 0, 0],
       [0, 0, 1, ..., 1, 0, 0],
       [0, 0, 0, ..., 1, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 1, 0, 1]])

In [13]:
mlb.classes_

array(['appeal_to_fear', 'bandwagon', 'cherry_picking', 'cliche',
       'euphoria', 'fud', 'glittering_generalities', 'loaded_language',
       'straw_man', 'whataboutism'], dtype=object)

### Spacy models setup

In [14]:
spacy.prefer_gpu()

True

In [15]:
# Download spacy models
!python -m spacy download uk_core_news_lg -q
!python -m spacy download ru_core_news_lg -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.2/231.2 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 115.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('uk_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.4/513.4 MB 4.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can 

In [16]:
# Load spaCy NER models
nlp_uk = spacy.load("uk_core_news_lg")
nlp_ru = spacy.load("ru_core_news_lg")

In [17]:
# Define stopwords sets (from spaCy)
stopwords_uk = nlp_uk.Defaults.stop_words
stopwords_ru = nlp_ru.Defaults.stop_words

In [18]:
# Define a dictionary for acronym expansion (Ukrainian & Russian)
acronym_dict = {
    "сша": "сполучені штати америки",
    "лол": "дуже смішно",
    "омг": "о боже",
    "бзв": "до вашого відома",
    "незнаю": "не знаю",  # Common typo fix

    "ссср": "союз советских социалистических республик",
    "нло": "неопознанный летающий объект",
    "мчс": "министерство чрезвычайных ситуаций",
}

### Preprocessing

In [19]:
def preprocess_text(text, lang):
    if lang == "ru":
        nlp = nlp_ru
        stopwords_set = stopwords_ru
    else:
        nlp = nlp_uk
        stopwords_set = stopwords_uk

    # Convert to lowercase
    text = text.lower()

    # Expand acronyms
    for acronym, expanded in acronym_dict.items():
        text = re.sub(r'\b' + re.escape(acronym) + r'\b', expanded, text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)

    # Remove emojis
    text = emoji.replace_emoji(text, replace='')

    # Remove extra spaces and newlines
    text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces with a single space
    text = text.replace('\n', ' ')    # Replace newline characters with a space
    text = text.strip()               # Remove leading/trailing spaces

    # Process text using spaCy (tokenization & lemmatization)
    doc = nlp(text)
    processed_words = [
        token.lemma_ for token in doc if token.text not in stopwords_set and not token.is_punct
    ]

    return ' '.join(processed_words)

In [20]:
df['clean_content'] = df.apply(lambda row: preprocess_text(row['content'], lang=row['lang']), axis=1)

## mBERT embeddings

In [34]:
MODEL_NAME_1 = "bert-base-multilingual-uncased"

In [ ]:
encoder1 = AutoModel.from_pretrained(MODEL_NAME_1).to(DEVICE)
tokenizer1 = BertTokenizer.from_pretrained(MODEL_NAME_1)

In [36]:
embeddings1 = []
for text in df['clean_content']:
    inputs = tokenizer1(text, return_tensors='pt', padding=True, truncation=True, max_length=512).to(DEVICE)
    outputs = encoder1(**inputs)
    embeddings1.append(outputs.last_hidden_state.mean(dim=1).cpu().detach().numpy())

In [37]:
embeddings1 = [row[0] for row in embeddings1]

In [38]:
X1 = pd.DataFrame(embeddings1)

## XLM-RoBERTa embeddings

In [21]:
MODEL_NAME_2 = "FacebookAI/xlm-roberta-large"

In [ ]:
encoder2 = AutoModel.from_pretrained(MODEL_NAME_2).to(DEVICE)
tokenizer2 = RobertaTokenizerFast.from_pretrained(MODEL_NAME_2, use_fast=True)

In [23]:
embeddings2 = []
for text in df['clean_content']:
    inputs = tokenizer2(text, return_tensors='pt', padding=True, truncation=True, max_length=512).to(DEVICE)
    outputs = encoder2(**inputs)
    embeddings2.append(outputs.last_hidden_state.mean(dim=1).cpu().detach().numpy())

In [26]:
embeddings2 = [row[0] for row in embeddings2]

In [27]:
X2 = pd.DataFrame(embeddings2)

# Modeling

## Initial

### mBert

In [39]:
# Split dataset
X_train1, X_test1, Y_train, Y_test = train_test_split(X1, Y, test_size=0.2, random_state=42)

In [40]:
models1 = {
    "Logistic Regression": MultiOutputClassifier(LogisticRegression(max_iter=1000)),
    "Random Forest": MultiOutputClassifier(RandomForestClassifier(n_estimators=100)),
    "XGBoost": MultiOutputClassifier(xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')),
    "LightGBM": MultiOutputClassifier(lgb.LGBMClassifier())
}

In [41]:
for name, model in models1.items():
    print(f"Training {name}...")
    model.fit(X_train1, Y_train)
    Y_pred = model.predict(X_test1)
    print(f"\n{name} Classification Report:\n")
    print(classification_report(Y_test, Y_pred, target_names=mlb.classes_))

Training Logistic Regression...

Logistic Regression Classification Report:

                         precision    recall  f1-score   support

         appeal_to_fear       0.32      0.10      0.16        58
              bandwagon       0.00      0.00      0.00        35
         cherry_picking       0.50      0.20      0.28        97
                 cliche       0.36      0.04      0.08        93
               euphoria       0.40      0.22      0.29        77
                    fud       0.52      0.19      0.27        75
glittering_generalities       0.65      0.46      0.54        97
        loaded_language       0.72      0.72      0.72       392
              straw_man       0.00      0.00      0.00        25
           whataboutism       0.00      0.00      0.00        34

              micro avg       0.64      0.39      0.49       983
              macro avg       0.35      0.19      0.23       983
           weighted avg       0.52      0.39      0.43       983
           

### XLM-RoBERTa

In [30]:
# Split dataset
X_train2, X_test2, Y_train, Y_test = train_test_split(X2, Y, test_size=0.2, random_state=42)

In [31]:
models2 = {
    "Logistic Regression": MultiOutputClassifier(LogisticRegression(max_iter=1000)),
    "Random Forest": MultiOutputClassifier(RandomForestClassifier(n_estimators=100)),
    "XGBoost": MultiOutputClassifier(xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')),
    "LightGBM": MultiOutputClassifier(lgb.LGBMClassifier())
}

In [33]:
for name, model in models2.items():
    print(f"Training {name}...")
    model.fit(X_train2, Y_train)
    Y_pred = model.predict(X_test2)
    print(f"\n{name} Classification Report:\n")
    print(classification_report(Y_test, Y_pred, target_names=mlb.classes_))

Training Logistic Regression...

Logistic Regression Classification Report:

                         precision    recall  f1-score   support

         appeal_to_fear       0.29      0.03      0.06        58
              bandwagon       0.00      0.00      0.00        35
         cherry_picking       0.43      0.16      0.24        97
                 cliche       0.25      0.01      0.02        93
               euphoria       0.52      0.17      0.25        77
                    fud       0.64      0.21      0.32        75
glittering_generalities       0.80      0.42      0.55        97
        loaded_language       0.76      0.75      0.75       392
              straw_man       0.00      0.00      0.00        25
           whataboutism       0.00      0.00      0.00        34

              micro avg       0.71      0.39      0.50       983
              macro avg       0.37      0.18      0.22       983
           weighted avg       0.55      0.39      0.43       983
           

## Hyperparameter tuning

Now we will try to modify models' hyperparameters and see whether the results can be better.

In [48]:
# Hyperparameter tuning for XGBoost
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', tree_method='gpu_hist',)
param_grid_xgb = {
    'learning_rate': [0.01, 0.1, 0.3],
    'n_estimators': [100, 300, 500],
    'max_depth': [3, 5, 7]
}

In [ ]:
grid_search_xgb = GridSearchCV(xgb_model, param_grid_xgb, cv=5, scoring='f1_macro', verbose=2, n_jobs=5)
grid_search_xgb.fit(X_train2, Y_train)

Fitting 5 folds for each of 27 candidates, totalling 135 fits


In [ ]:
print("Best parameters for XGBoost:", grid_search_xgb.best_params_)

y_pred_xgb = grid_search_xgb.best_estimator_.predict(X_test)
print("\nXGBoost Classification Report:\n")
print(classification_report(Y_test, y_pred_xgb, target_names=mlb.classes_))

In [ ]:
# Hyperparameter tuning for LightGBM
lgb_model = lgb.LGBMClassifier(device='gpu')
param_grid_lgb = {
    'learning_rate': [0.01, 0.1, 0.3],
    'n_estimators': [100, 300, 500],
    'num_leaves': [31, 50, 70]
}

In [ ]:
grid_search_lgb = GridSearchCV(lgb_model, param_grid_lgb, cv=5, scoring='f1_macro', verbose=2, n_jobs=-1)
grid_search_lgb.fit(X_train.toarray(), Y_train)

In [ ]:
print("Best parameters for LightGBM:", grid_search_lgb.best_params_)

y_pred_lgb = grid_search_lgb.best_estimator_.predict(X_test)
print("\nLightGBM Classification Report:\n")
print(classification_report(Y_test, y_pred_lgb, target_names=mlb.classes_))


Fitting 3 folds for each of 27 candidates, totalling 81 fits


BrokenProcessPool: A task has failed to un-serialize. Please ensure that the arguments of the function are all picklable.